# Modeling LGBM 개선 노트북

기존 `new.ipynb`의 데이터셋 비교 구조를 유지하면서, LGBMRegressor + TimeSeriesSplit + Optuna 튜닝을 추가한 버전이다.

핵심 수정:
- 시계열 데이터이므로 랜덤 split 금지
- train 내부에서 TimeSeriesSplit CV
- 마지막 20%는 최종 holdout test
- baseline과 LGBM을 같이 비교
- `_with_dummies` 데이터셋까지 비교
- Optuna 튜닝 결과, 예측값, feature importance 저장

## 수정 반영 메모

- Optuna 튜닝 중 `max_depth=2`가 선택되면 `max_num_leaves = 2^2 - 1 = 3`이 됩니다.
- 기존 코드에서는 `num_leaves`를 `7 ~ max_num_leaves` 범위에서 고르게 해서 `7 ~ 3` 범위 오류가 발생했습니다.
- 그래서 `max_depth` 탐색 범위를 `3 ~ 7`로 수정했습니다.
- 실행할 때는 위에서부터 다시 실행하거나, 최소한 함수 정의 셀을 먼저 다시 실행한 뒤 Optuna 튜닝 셀을 실행하면 됩니다.


In [ ]:
# 필요 시 최초 1회만 실행
# !pip install lightgbm optuna scikit-learn pandas numpy

In [ ]:
# Modeling_LGBM_improved.py
# 목적:
# - 기존 OLS/Ridge/Lasso 비교 구조에 LGBMRegressor 추가
# - 시계열 데이터 특성을 반영해 TimeSeriesSplit 기반 CV + 마지막 20% holdout 평가
# - Optuna 튜닝, baseline 비교, feature importance 저장까지 한 번에 수행
#
# 실행 위치 권장:
#   repo/src 안에서 실행
#
# 필요 패키지:
#   pip install lightgbm optuna scikit-learn pandas numpy

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
except ImportError as e:
    raise ImportError(
        "lightgbm이 설치되어 있지 않습니다. 먼저 `pip install lightgbm` 실행 후 다시 돌려주세요."
    ) from e


# =========================
# 0. 설정
# =========================

RANDOM_STATE = 42
DATA_DIR = Path("../data/Finance_Final")
OUTPUT_DIR = Path("../outputs/lgbm_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "oil_diff_target"
DATE_COL = "date"

# 기존 4개 데이터셋 + 이벤트 더미 포함 데이터셋까지 같이 비교
DATASET_FILES = {
    "dataset1_raw_only": "dataset1_raw_only.csv",
    "dataset2_raw_plus_derived": "dataset2_raw_plus_derived.csv",
    "dataset3_diff_only": "dataset3_diff_only.csv",
    "dataset4_derived_full": "dataset4_derived_full.csv",
    "dataset1_raw_only_with_dummies": "dataset1_raw_only_with_dummies.csv",
    "dataset2_raw_plus_derived_with_dummies": "dataset2_raw_plus_derived_with_dummies.csv",
    "dataset3_diff_only_with_dummies": "dataset3_diff_only_with_dummies.csv",
    "dataset4_derived_full_with_dummies": "dataset4_derived_full_with_dummies.csv",
}

TEST_SIZE = 0.20
N_SPLITS = 5

# 시간이 부족하면 20~30, 발표용 최종은 50~100 정도 권장
RUN_OPTUNA = True
N_TRIALS = 50

# 전체 데이터셋을 다 튜닝하면 오래 걸릴 수 있어서 1차 기본 LGBM 결과 상위 몇 개만 튜닝
TOP_K_FOR_TUNING = 3


# =========================
# 1. 유틸 함수
# =========================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def directional_accuracy(y_true, y_pred, threshold=0.0, exclude_flat=True):
    """
    방향 정확도:
    - y_true와 y_pred의 부호가 같은 비율
    - threshold > 0이면 예측값이 너무 작은 경우 0 = no signal로 처리
    - exclude_flat=True이면 실제값 또는 예측값이 0인 행은 방향 평가에서 제외
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    y_true_sign = np.sign(y_true)

    if threshold > 0:
        y_pred_sign = np.where(y_pred > threshold, 1, np.where(y_pred < -threshold, -1, 0))
    else:
        y_pred_sign = np.sign(y_pred)

    if exclude_flat:
        mask = (y_true_sign != 0) & (y_pred_sign != 0)
        if mask.sum() == 0:
            return np.nan
        return float(np.mean(y_true_sign[mask] == y_pred_sign[mask]))

    return float(np.mean(y_true_sign == y_pred_sign))


def evaluate_predictions(y_true, y_pred, dataset, model, n_train, n_test, feature_count):
    return {
        "dataset": dataset,
        "model": model,
        "n_train": n_train,
        "n_test": n_test,
        "feature_count": feature_count,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Directional_Accuracy": directional_accuracy(y_true, y_pred),
        "Directional_Accuracy_thr_0.05": directional_accuracy(y_true, y_pred, threshold=0.05),
    }


def load_datasets():
    datasets = {}

    for name, file in DATASET_FILES.items():
        path = DATA_DIR / file

        if not path.exists():
            print(f"[skip] 파일 없음: {path}")
            continue

        df = pd.read_csv(path)

        if DATE_COL in df.columns:
            df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
            df = df.sort_values(DATE_COL).reset_index(drop=True)

        datasets[name] = df

    print("데이터 불러오기 완료")
    for name, df in datasets.items():
        date_range = ""
        if DATE_COL in df.columns:
            date_range = f" / {df[DATE_COL].min().date()} ~ {df[DATE_COL].max().date()}"
        print(f"- {name}: {df.shape}{date_range}")

    return datasets


def prepare_xy(df, target_col=TARGET_COL, date_col=DATE_COL):
    """
    모델용 X, y 생성.
    - 숫자형 변수만 사용
    - date, target, target/shift/future가 이름에 들어간 잠재 누수 컬럼 제거
    - 단, 현재 target_col은 y로만 사용
    """
    df = df.copy()

    if target_col not in df.columns:
        raise ValueError(f"target_col={target_col} 컬럼이 없습니다.")

    # 날짜 정렬
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.sort_values(date_col).reset_index(drop=True)

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    leakage_keywords = ["target", "future", "shift"]
    feature_cols = []

    for c in numeric_cols:
        if c == target_col:
            continue

        lower_c = c.lower()

        # oil_return_shift, next_return 같은 미래정보 의심 컬럼 방지
        if any(k in lower_c for k in leakage_keywords):
            continue

        feature_cols.append(c)

    keep_cols = [target_col] + feature_cols
    if date_col in df.columns:
        keep_cols = [date_col] + keep_cols

    model_df = df[keep_cols].dropna().reset_index(drop=True)

    X = model_df[feature_cols]
    y = model_df[target_col]

    return model_df, X, y, feature_cols


def time_holdout_split(model_df, X, y, test_size=TEST_SIZE):
    split_idx = int(len(model_df) * (1 - test_size))

    train_df = model_df.iloc[:split_idx].reset_index(drop=True)
    test_df = model_df.iloc[split_idx:].reset_index(drop=True)

    X_train = X.iloc[:split_idx].reset_index(drop=True)
    X_test = X.iloc[split_idx:].reset_index(drop=True)
    y_train = y.iloc[:split_idx].reset_index(drop=True)
    y_test = y.iloc[split_idx:].reset_index(drop=True)

    return train_df, test_df, X_train, X_test, y_train, y_test


def make_lgbm(params=None):
    """
    기본값은 과적합 방지 쪽으로 약간 보수적으로 설정.
    Optuna가 params를 넘기면 해당 값으로 업데이트.
    """
    base_params = {
        "objective": "regression",
        "n_estimators": 4000,
        "learning_rate": 0.01,
        "num_leaves": 15,
        "max_depth": 4,
        "min_child_samples": 80,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "min_split_gain": 0.0,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
        "force_col_wise": True,
    }

    if params is not None:
        base_params.update(params)

    return LGBMRegressor(**base_params)


def fit_lgbm_with_last_valid(X_train, y_train, params=None, valid_ratio=0.2):
    """
    train 내부 마지막 구간을 validation으로 두고 early stopping 수행.
    test holdout은 건드리지 않음.
    """
    valid_size = max(int(len(X_train) * valid_ratio), 1)

    X_tr = X_train.iloc[:-valid_size]
    y_tr = y_train.iloc[:-valid_size]
    X_val = X_train.iloc[-valid_size:]
    y_val = y_train.iloc[-valid_size:]

    model = make_lgbm(params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            early_stopping(stopping_rounds=100, verbose=False),
            log_evaluation(period=0),
        ],
    )
    return model


def cv_lgbm_score(X_train, y_train, params=None, n_splits=N_SPLITS):
    """
    train 구간 내부에서만 TimeSeriesSplit CV.
    튜닝 기준은 CV RMSE 평균.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)

    rows = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = make_lgbm(params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            callbacks=[
                early_stopping(stopping_rounds=100, verbose=False),
                log_evaluation(period=0),
            ],
        )

        pred = model.predict(X_val)

        rows.append({
            "fold": fold,
            "MAE": mean_absolute_error(y_val, pred),
            "RMSE": rmse(y_val, pred),
            "R2": r2_score(y_val, pred),
            "Directional_Accuracy": directional_accuracy(y_val, pred),
        })

    return pd.DataFrame(rows)


def baseline_predictions(X_test, y_train):
    """
    LGBM이 실제로 유용한지 비교하기 위한 baseline.
    - zero: 변화 없음
    - train_mean: 학습구간 평균 변화량
    - train_median: 학습구간 중앙값 변화량
    - lag1_persistence: oil_diff_lag1이 있으면 전일 변화 지속 가정
    """
    baselines = {
        "naive_zero": np.zeros(len(X_test)),
        "naive_train_mean": np.repeat(y_train.mean(), len(X_test)),
        "naive_train_median": np.repeat(y_train.median(), len(X_test)),
    }

    if "oil_diff_lag1" in X_test.columns:
        baselines["naive_lag1_persistence"] = X_test["oil_diff_lag1"].values

    return baselines


def tune_lgbm_optuna(X_train, y_train, n_trials=N_TRIALS, n_splits=N_SPLITS):
    try:
        import optuna
    except ImportError as e:
        raise ImportError(
            "optuna가 설치되어 있지 않습니다. 먼저 `pip install optuna` 실행 후 다시 돌려주세요."
        ) from e

    def objective(trial):
        max_depth = trial.suggest_int("max_depth", 3, 7)  # fixed: max_depth=2이면 num_leaves 상한이 3이라 오류 발생

        # LightGBM은 leaf-wise 방식이라 num_leaves를 너무 크게 잡으면 과적합 위험이 커짐.
        max_num_leaves = min(2 ** max_depth - 1, 63)

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 7, max_num_leaves),
            "max_depth": max_depth,
            "min_child_samples": trial.suggest_int("min_child_samples", 30, 250),
            "subsample": trial.suggest_float("subsample", 0.55, 1.0),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.3),
        }

        cv_result = cv_lgbm_score(X_train, y_train, params=params, n_splits=n_splits)
        return cv_result["RMSE"].mean()

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )

    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    return study.best_params, study.best_value, study


def run_dataset(dataset_name, df, params=None, model_label="LGBM_default", save_detail=True):
    model_df, X, y, feature_cols = prepare_xy(df)
    train_df, test_df, X_train, X_test, y_train, y_test = time_holdout_split(model_df, X, y)

    results = []

    # Baseline 평가
    for baseline_name, pred in baseline_predictions(X_test, y_train).items():
        results.append(
            evaluate_predictions(
                y_test, pred,
                dataset=dataset_name,
                model=baseline_name,
                n_train=len(y_train),
                n_test=len(y_test),
                feature_count=len(feature_cols),
            )
        )

    # LGBM 평가
    cv_result = cv_lgbm_score(X_train, y_train, params=params)
    model = fit_lgbm_with_last_valid(X_train, y_train, params=params)
    pred_test = model.predict(X_test)

    row = evaluate_predictions(
        y_test, pred_test,
        dataset=dataset_name,
        model=model_label,
        n_train=len(y_train),
        n_test=len(y_test),
        feature_count=len(feature_cols),
    )
    row["CV_RMSE_mean"] = cv_result["RMSE"].mean()
    row["CV_RMSE_std"] = cv_result["RMSE"].std()
    row["best_iteration"] = getattr(model, "best_iteration_", None)
    results.append(row)

    result_df = pd.DataFrame(results)

    if save_detail:
        pred_df = pd.DataFrame({
            "date": test_df[DATE_COL].values if DATE_COL in test_df.columns else np.arange(len(y_test)),
            "y_true": y_test.values,
            "y_pred": pred_test,
            "pred_sign": np.sign(pred_test),
            "true_sign": np.sign(y_test.values),
        })
        pred_df.to_csv(OUTPUT_DIR / f"prediction_{dataset_name}_{model_label}.csv", index=False)

        fi_df = pd.DataFrame({
            "feature": feature_cols,
            "importance_gain": model.booster_.feature_importance(importance_type="gain"),
            "importance_split": model.booster_.feature_importance(importance_type="split"),
        }).sort_values("importance_gain", ascending=False)
        fi_df.to_csv(OUTPUT_DIR / f"feature_importance_{dataset_name}_{model_label}.csv", index=False)

        cv_result.to_csv(OUTPUT_DIR / f"cv_{dataset_name}_{model_label}.csv", index=False)

    return result_df

In [ ]:
datasets = load_datasets()

In [ ]:
# 1차: 모든 데이터셋에 대해 baseline + 기본 LGBM 비교
all_default_results = []

for name, df in datasets.items():
    print(f"\n===== {name} =====")
    result = run_dataset(name, df, params=None, model_label="LGBM_default", save_detail=True)
    display(result)
    all_default_results.append(result)

default_compare = pd.concat(all_default_results, ignore_index=True)
default_compare.to_csv(OUTPUT_DIR / "lgbm_default_compare.csv", index=False)

default_compare.sort_values(["RMSE", "Directional_Accuracy"], ascending=[True, False])

In [ ]:
# 튜닝 후보: 기본 LGBM의 CV_RMSE가 좋은 상위 데이터셋
lgbm_only = (
    default_compare[default_compare["model"] == "LGBM_default"]
    .sort_values("CV_RMSE_mean")
    .reset_index(drop=True)
)

tuning_targets = lgbm_only["dataset"].head(TOP_K_FOR_TUNING).tolist()
tuning_targets

In [ ]:
# 2차: Optuna 튜닝
# 시간이 오래 걸리면 N_TRIALS를 20~30으로 줄여도 됨
tuned_rows = []
best_param_rows = []

if RUN_OPTUNA:
    for name in tuning_targets:
        print(f"\n===== tuning: {name} =====")
        df = datasets[name]

        model_df, X, y, feature_cols = prepare_xy(df)
        _, _, X_train, _, y_train, _ = time_holdout_split(model_df, X, y)

        best_params, best_cv_rmse, study = tune_lgbm_optuna(
            X_train,
            y_train,
            n_trials=N_TRIALS,
            n_splits=N_SPLITS,
        )

        best_param_rows.append({
            "dataset": name,
            "best_cv_rmse": best_cv_rmse,
            **best_params,
        })

        tuned_result = run_dataset(
            name,
            df,
            params=best_params,
            model_label="LGBM_optuna",
            save_detail=True,
        )

        display(tuned_result)
        tuned_rows.append(tuned_result[tuned_result["model"] == "LGBM_optuna"])

best_params_df = pd.DataFrame(best_param_rows)
best_params_df.to_csv(OUTPUT_DIR / "optuna_best_params.csv", index=False)
best_params_df

In [ ]:
# 최종 비교표
final_compare = default_compare.copy()

if len(tuned_rows) > 0:
    final_compare = pd.concat([final_compare] + tuned_rows, ignore_index=True)

final_compare = final_compare.sort_values(
    ["RMSE", "Directional_Accuracy"],
    ascending=[True, False]
).reset_index(drop=True)

final_compare.to_csv(OUTPUT_DIR / "lgbm_final_compare.csv", index=False)
final_compare.head(20)

## 결과 해석 기준

- RMSE 기준 1등 모델이 baseline보다 낮아야 수치 예측력이 있다고 볼 수 있다.
- 방향 정확도는 0.5보다 얼마나 안정적으로 높은지 확인한다.
- RMSE는 좋은데 방향 정확도가 낮으면 투자 방향 판단에는 약하다.
- 방향 정확도는 좋은데 RMSE가 나쁘면 상승/하락 방향만 약간 맞추고 크기 예측은 약한 모델이다.
- feature importance는 인과관계가 아니라 예측 과정에서 많이 쓰인 변수로 해석한다.